# Model PD Css Cross - single use notebook

Linear version of `pd_css_cross_final_notebook.ipynb`. The code is expanded for one run through the model pipeline, with no user helper function blocks.


## Konfiguracja
Importy, ścieżki, stałe modelu oraz reguły wyboru zmiennych.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from openpyxl import load_workbook
import statsmodels.api as sm

PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "abt_app.sas7bdat").exists()
)

DATA_PATH = PROJECT_ROOT / "abt_app.sas7bdat"
OUTPUT_DIR = PROJECT_ROOT / "outputs/pd_css_cross"
SAS_PATH = OUTPUT_DIR / "scoring_code.sas"
GINI_CURVES_TEMPLATE_PATH = PROJECT_ROOT / "gini_curves_template.xlsx"
GINI_CURVES_OUTPUT_PATH = OUTPUT_DIR / "gini_curves_model.xlsx"

MODEL_ID = "PD_CSS_CROSS"
TARGET = "default_cross12"
ID_COLUMN = "aid"
PERIOD_COLUMN = "period"
PD_COLUMN = "PD_CSS_CROSS"
SCORE_COLUMN = "SCORE_PD_CSS_CROSS"

PERIOD_FROM = "197501"
PERIOD_TO = "198712"
RANDOM_STATE = 1234
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
MIN_HOLDOUT_ROWS = 250
LOGISTIC_C = 0.3
FEATURE_SELECTION_MODE = "all_leakage_safe"
NUMERIC_ADD_INDICATORS = False
ONEHOT_DROP = "first"
CORRELATION_PRUNING_THRESHOLD = 0.75
VIF_PRUNING_THRESHOLD = 10.0
MAX_VIF_PRUNING_STEPS = 50

APPLICATION_PRODUCT = "all"
DECISION = "A"

CANDIDATE_PREFIXES = ("app", "act", "agr", "ags")
APPLICATION_FEATURES = ("product",)
MAX_MISSING_RATE = 0.98
MAX_CATEGORY_LEVELS = 50
MAX_DOMINANT_SHARE = 0.995
MIN_UNIVARIATE_GINI = 0.02
ANALYSIS_MAX_BINS = 4
ANALYSIS_MIN_BIN_SHARE = 0.05
MAX_VARIABLE_REPORT_SHEETS = 50
ASB_GINI_TRAIN_MIN = 0.05
ASB_RELATIVE_GINI_MAX = 0.2
ASB_PSI_MAX = 0.1
ASB_PSI_BAD_MAX = 0.1
EPSILON = 0.0001
SYMBOL_MISSING = "Missing"
SYMBOL_OTHER = "<OTHERS>"
SCORECARD_FACTOR = 20 / np.log(2)
SCORECARD_BASE_POINTS = 300
LEAKAGE_TOKENS = (
    "default",
    "cross_response",
    "cross_after",
    "cross_aid",
    "pd_",
    "score_",
)

ENGINEERED_RATIOS = {
    "eng_app_loan_to_income": ("app_loan_amount", "app_income"),
    "eng_app_installment_to_income": ("app_installment", "app_income"),
    "eng_app_spending_to_income": ("app_spendings", "app_income"),
    "eng_act_loaninc_to_income": ("act_loaninc", "app_income"),
    "eng_act_cc_to_income": ("act_cc", "app_income"),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)




## Krok 1: próba, zmienne i podział OOT
Wczytanie ABT, brak filtra po `cross_response`, wybór cech leakage-safe, cechy techniczne oraz podział train/validation/test po czasie.


### Wczytanie i filtr próby


In [ ]:
df = pd.read_sas(DATA_PATH, encoding="LATIN2")
df[PERIOD_COLUMN] = df[PERIOD_COLUMN].astype(str)
for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].str.strip().replace("", np.nan)

application_mask = (
    df[PERIOD_COLUMN].between(PERIOD_FROM, PERIOD_TO)
    & df["decision"].eq(DECISION)
)
target_mask = application_mask & df[TARGET].notna()
if target_mask.sum() == 0:
    raise ValueError(f"No rows found with non-missing {TARGET}.")

sample = df.loc[target_mask].reset_index(drop=True)
application_product_counts = df.loc[application_mask, "product"].value_counts().sort_index()
target_known_sample_product_counts = sample["product"].value_counts().sort_index()
sample_note = (
    f"PD Css Cross sample: decision={DECISION}, product={APPLICATION_PRODUCT}, "
    f"target={TARGET} not missing, no cross-response prefilter."
)
sample.attrs["sample_note"] = sample_note
sample.attrs["sample_audit"] = {
    "model_id": MODEL_ID,
    "target": TARGET,
    "period_from": PERIOD_FROM,
    "period_to": PERIOD_TO,
    "decision": DECISION,
    "product": APPLICATION_PRODUCT,
    "application_product_counts": ";".join(f"{product}:{count}" for product, count in application_product_counts.items()),
    "target_known_sample_product_counts": ";".join(f"{product}:{count}" for product, count in target_known_sample_product_counts.items()),
    "application_rows": int(application_mask.sum()),
    "target_non_missing_rows": int(target_mask.sum()),
    "duplicate_id_rows": int(sample.duplicated(ID_COLUMN).sum()),
    "event_rate": float(sample[TARGET].mean()),
    "sample_note": sample_note,
}



### Lista zmiennych wejściowych


In [ ]:
raw_features = [
    column
    for column in sample.columns
    if column[:3].lower() in CANDIDATE_PREFIXES
    and not any(token in column.lower() for token in LEAKAGE_TOKENS)
]
for column in reversed(APPLICATION_FEATURES):
    if column in sample.columns and column not in raw_features:
        raw_features.insert(0, column)



### Cechy techniczne i target


In [ ]:
X_all = sample[raw_features].copy()
for new_column, (numerator, denominator) in ENGINEERED_RATIOS.items():
    if numerator in X_all.columns and denominator in X_all.columns:
        X_all[new_column] = X_all[numerator] / X_all[denominator].replace(0, np.nan)
X_all["eng_missing_count"] = X_all.isna().sum(axis=1)
X_all = X_all.replace([np.inf, -np.inf], np.nan)
y = sample[TARGET].astype(int)



### Podział train / validation / test


In [ ]:
sample_periods = sample[[PERIOD_COLUMN]].copy()
sample_periods["row_number"] = np.arange(sample.shape[0])
period_counts = sample_periods.groupby(PERIOD_COLUMN).size().sort_index().rename("n_obs").reset_index()

test_target = max(MIN_HOLDOUT_ROWS, int(round(sample.shape[0] * TEST_FRACTION)))
validation_target = max(MIN_HOLDOUT_ROWS, int(round(sample.shape[0] * VALIDATION_FRACTION)))
period_counts["cum_from_end"] = period_counts["n_obs"][::-1].cumsum()[::-1]
test_periods = period_counts.loc[period_counts["cum_from_end"] <= test_target, PERIOD_COLUMN]
if test_periods.empty:
    test_periods = period_counts.tail(1)[PERIOD_COLUMN]
test_start = test_periods.min()

train_validation_periods = period_counts.loc[period_counts[PERIOD_COLUMN] < test_start].copy()
train_validation_periods["cum_from_end"] = train_validation_periods["n_obs"][::-1].cumsum()[::-1]
validation_periods = train_validation_periods.loc[
    train_validation_periods["cum_from_end"] <= validation_target, PERIOD_COLUMN
]
if validation_periods.empty:
    validation_periods = train_validation_periods.tail(1)[PERIOD_COLUMN]
validation_start = validation_periods.min()

masks = {
    "train": sample[PERIOD_COLUMN] < validation_start,
    "validation": sample[PERIOD_COLUMN].between(validation_start, test_start, inclusive="left"),
    "test": sample[PERIOD_COLUMN] >= test_start,
}

if any(int(mask.sum()) == 0 for mask in masks.values()):
    raise ValueError("Temporal split produced an empty train, validation, or test sample.")
for split_name, mask in masks.items():
    if y.loc[mask].nunique() < 2:
        raise ValueError(f"Split {split_name} has only one target class.")



### Zapis definicji podziału


In [ ]:
split_definition = {
    "validation_start": validation_start,
    "test_start": test_start,
    "train_rows": int(masks["train"].sum()),
    "validation_rows": int(masks["validation"].sum()),
    "test_rows": int(masks["test"].sum()),
}

pd.DataFrame([split_definition]).to_csv(OUTPUT_DIR / "split_definition.csv", index=False)
sample.attrs["sample_note"], X_all.shape, y.mean(), split_definition



## Krok 2: diagnostyka zmiennych, korelacja i VIF
Selekcja jakościowa, usunięcie silnych korelacji oraz ograniczenie VIF bez helperów.


### Diagnostyka jakości zmiennych


In [ ]:
y_train = y.loc[masks["train"]]
quality_rows = []
for column in X_all.columns:
    train_col = X_all.loc[masks["train"], column]
    missing_rate = float(train_col.isna().mean())
    nunique = int(train_col.nunique(dropna=True))
    top_share = float(train_col.value_counts(dropna=False, normalize=True).iloc[0])
    is_numeric = pd.api.types.is_numeric_dtype(train_col)

    if is_numeric:
        score = pd.to_numeric(train_col, errors="coerce")
        median = score.median()
        score = score.fillna(median)
    else:
        grouped = pd.DataFrame({"value": train_col.fillna("__MISSING__"), "target": y_train}).groupby("value")["target"].mean()
        global_rate = float(y_train.mean())
        score = train_col.fillna("__MISSING__").map(grouped).fillna(global_rate)

    if pd.Series(score).nunique(dropna=True) < 2:
        univariate_gini = 0.0
    else:
        univariate_gini = float(abs(2.0 * roc_auc_score(y_train, score) - 1.0))

    rejection_reasons = []
    if missing_rate > MAX_MISSING_RATE:
        rejection_reasons.append("too_many_missing")
    if nunique <= 1:
        rejection_reasons.append("constant_or_empty")
    if top_share > MAX_DOMINANT_SHARE:
        rejection_reasons.append("dominant_single_value")
    if not is_numeric and nunique > MAX_CATEGORY_LEVELS:
        rejection_reasons.append("too_many_category_levels")
    if univariate_gini < MIN_UNIVARIATE_GINI:
        rejection_reasons.append("weak_univariate_gini")

    quality_rows.append(
        {
            "feature": column,
            "dtype": str(train_col.dtype),
            "is_numeric": bool(is_numeric),
            "missing_rate_train": missing_rate,
            "nunique_train": nunique,
            "top_value_share_train": top_share,
            "univariate_abs_gini_train": univariate_gini,
            "selected": not rejection_reasons,
            "rejection_reason": ";".join(rejection_reasons),
        }
    )



### Wybór zmiennych po jakości


In [ ]:
feature_report = pd.DataFrame(quality_rows).sort_values(
    ["selected", "univariate_abs_gini_train"], ascending=[False, False]
)
quality_selected_features = feature_report.loc[feature_report["selected"], "feature"].tolist()

if FEATURE_SELECTION_MODE == "quality_screen":
    selected_features = quality_selected_features
elif FEATURE_SELECTION_MODE == "all_leakage_safe":
    selected_features = raw_features.copy()
else:
    raise ValueError(f"Unknown FEATURE_SELECTION_MODE: {FEATURE_SELECTION_MODE}")



### Korelacje surowych zmiennych numerycznych


In [ ]:
pre_correlation_features = selected_features.copy()
quality = feature_report.set_index("feature")
candidate_features = list(pre_correlation_features)
numeric_features = [feature for feature in candidate_features if pd.api.types.is_numeric_dtype(X_all[feature])]
categorical_features = [feature for feature in candidate_features if feature not in numeric_features]
correlation_rows = []

if len(numeric_features) < 2:
    preprocessed_features = candidate_features
else:
    train_numeric = X_all.loc[masks["train"], numeric_features].apply(pd.to_numeric, errors="coerce")
    train_numeric = train_numeric.fillna(train_numeric.median())
    corr = train_numeric.corr().abs()
    priority = quality.loc[numeric_features, "univariate_abs_gini_train"].fillna(0.0).sort_values(ascending=False)
    kept = []
    dropped = set()
    for feature in priority.index:
        if feature in dropped:
            continue
        kept.append(feature)
        correlated = corr.index[(corr[feature] >= CORRELATION_PRUNING_THRESHOLD) & (corr.index != feature)]
        for other in correlated:
            if other in dropped or other in kept:
                continue
            dropped.add(other)
            correlation_rows.append(
                {
                    "stage": "raw_numeric_correlation",
                    "dropped_feature": other,
                    "kept_feature": feature,
                    "dropped_encoded_term": other,
                    "kept_encoded_term": feature,
                    "abs_correlation": float(corr.loc[feature, other]),
                    "dropped_univariate_abs_gini": float(quality.loc[other, "univariate_abs_gini_train"]),
                    "kept_univariate_abs_gini": float(quality.loc[feature, "univariate_abs_gini_train"]),
                    "threshold": CORRELATION_PRUNING_THRESHOLD,
                }
            )
    kept_numeric = [feature for feature in numeric_features if feature not in dropped]
    preprocessed_features = [feature for feature in candidate_features if feature in categorical_features or feature in kept_numeric]



### Korelacje po preprocessingu


In [ ]:
if len(preprocessed_features) >= 2:
    numeric_columns = X_all[preprocessed_features].select_dtypes(include="number").columns.tolist()
    categorical_columns = [column for column in preprocessed_features if column not in numeric_columns]
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=NUMERIC_ADD_INDICATORS)),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, drop=ONEHOT_DROP)),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[("num", numeric_pipe, numeric_columns), ("cat", categorical_pipe, categorical_columns)]
    )
    preprocessor.fit(X_all.loc[masks["train"], preprocessed_features])
    design = preprocessor.transform(X_all.loc[masks["train"], preprocessed_features])
    if hasattr(design, "toarray"):
        design = design.toarray()
    design = np.asarray(design, dtype=float)
    feature_names = np.array(preprocessor.get_feature_names_out())
    numeric_columns = list(preprocessor.transformers_[0][2])
    categorical_columns = list(preprocessor.transformers_[1][2])
    raw_by_term = []
    for name in feature_names:
        if name.startswith("num__missingindicator_"):
            raw_by_term.append(name.replace("num__missingindicator_", "", 1))
        elif name.startswith("num__"):
            raw_by_term.append(name.replace("num__", "", 1))
        elif name.startswith("cat__"):
            payload = name.replace("cat__", "", 1)
            mapped = name
            for column in sorted(categorical_columns, key=len, reverse=True):
                if payload == column or payload.startswith(f"{column}_"):
                    mapped = column
                    break
            raw_by_term.append(mapped)
        else:
            raw_by_term.append(name)
    raw_by_term = np.array(raw_by_term)

    std = design.std(axis=0)
    keep = std > 1e-12
    design = design[:, keep]
    feature_names = feature_names[keep]
    raw_by_term = raw_by_term[keep]
    std = std[keep]
    design = (design - design.mean(axis=0)) / std

    if design.shape[1] >= 2:
        encoded_corr = np.abs(np.corrcoef(design, rowvar=False))
        encoded_corr = np.nan_to_num(encoded_corr, nan=0.0, posinf=1.0, neginf=1.0)
        dropped_raw = set()
        raw_priority = quality["univariate_abs_gini_train"].fillna(0.0).to_dict()
        pair_rows = []
        for pair in np.argwhere(np.triu(encoded_corr, k=1) >= CORRELATION_PRUNING_THRESHOLD):
            pair_rows.append({"left_idx": int(pair[0]), "right_idx": int(pair[1]), "abs_correlation": float(encoded_corr[pair[0], pair[1]])})
        pair_rows = pd.DataFrame(pair_rows).sort_values("abs_correlation", ascending=False) if pair_rows else pd.DataFrame(columns=["left_idx", "right_idx", "abs_correlation"])
        for _, pair_row in pair_rows.iterrows():
            left_idx = int(pair_row["left_idx"])
            right_idx = int(pair_row["right_idx"])
            left_raw = raw_by_term[left_idx]
            right_raw = raw_by_term[right_idx]
            if left_raw == right_raw or left_raw in dropped_raw or right_raw in dropped_raw:
                continue
            left_score = raw_priority.get(left_raw, 0.0)
            right_score = raw_priority.get(right_raw, 0.0)
            if left_score > right_score:
                kept_raw, dropped = left_raw, right_raw
                kept_term, dropped_term = feature_names[left_idx], feature_names[right_idx]
                kept_score, dropped_score = left_score, right_score
            elif right_score > left_score:
                kept_raw, dropped = right_raw, left_raw
                kept_term, dropped_term = feature_names[right_idx], feature_names[left_idx]
                kept_score, dropped_score = right_score, left_score
            else:
                kept_raw, dropped = sorted([left_raw, right_raw])[0], sorted([left_raw, right_raw])[1]
                kept_term = feature_names[left_idx] if kept_raw == left_raw else feature_names[right_idx]
                dropped_term = feature_names[right_idx] if dropped == right_raw else feature_names[left_idx]
                kept_score = raw_priority.get(kept_raw, 0.0)
                dropped_score = raw_priority.get(dropped, 0.0)
            dropped_raw.add(dropped)
            correlation_rows.append(
                {
                    "stage": "encoded_correlation",
                    "dropped_feature": dropped,
                    "kept_feature": kept_raw,
                    "dropped_encoded_term": dropped_term,
                    "kept_encoded_term": kept_term,
                    "abs_correlation": float(encoded_corr[left_idx, right_idx]),
                    "dropped_univariate_abs_gini": float(dropped_score),
                    "kept_univariate_abs_gini": float(kept_score),
                    "threshold": CORRELATION_PRUNING_THRESHOLD,
                }
            )
        selected_features = [feature for feature in preprocessed_features if feature not in dropped_raw]
    else:
        selected_features = preprocessed_features
else:
    selected_features = preprocessed_features

correlation_pruning = pd.DataFrame(correlation_rows)
if not correlation_pruning.empty:
    correlation_pruning = correlation_pruning.sort_values(
        ["abs_correlation", "dropped_feature"], ascending=[False, True]
    ).reset_index(drop=True)



### Przygotowanie redukcji VIF


In [ ]:
pre_vif_features = selected_features.copy()
selected_features = list(pre_vif_features)
vif_pruning_rows = []
status_rank = {"ok": 0, "single_active_term": 0, "undefined": 1, "near_exact_collinearity": 2}



### Iteracyjne usuwanie wysokiego VIF


In [ ]:
for step in range(1, MAX_VIF_PRUNING_STEPS + 1):
    if len(selected_features) <= 2:
        break

    numeric_columns = X_all[selected_features].select_dtypes(include="number").columns.tolist()
    categorical_columns = [column for column in selected_features if column not in numeric_columns]
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=NUMERIC_ADD_INDICATORS)),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, drop=ONEHOT_DROP)),
        ]
    )
    preprocess = ColumnTransformer(
        transformers=[("num", numeric_pipe, numeric_columns), ("cat", categorical_pipe, categorical_columns)]
    )
    clf = LogisticRegression(
        penalty="l1", solver="liblinear", C=LOGISTIC_C, max_iter=2000, random_state=RANDOM_STATE
    )
    fitted_for_vif = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])
    fitted_for_vif.fit(X_all.loc[masks["train"], selected_features], y.loc[masks["train"]])

    preprocess = fitted_for_vif.named_steps["preprocess"]
    classifier = fitted_for_vif.named_steps["model"]
    feature_names = np.array(preprocess.get_feature_names_out())
    coefficients = classifier.coef_[0]
    selected_mask = np.ones_like(coefficients, dtype=bool)
    design = preprocess.transform(X_all.loc[masks["train"], selected_features])
    design = design[:, selected_mask]
    if hasattr(design, "toarray"):
        design = design.toarray()
    design = np.asarray(design, dtype=float)
    selected_encoded_features = feature_names[selected_mask]
    selected_coefficients = coefficients[selected_mask]
    numeric_columns = list(preprocess.transformers_[0][2])
    categorical_columns = list(preprocess.transformers_[1][2])
    raw_features_for_terms = []
    for name in selected_encoded_features:
        if name.startswith("num__missingindicator_"):
            raw_features_for_terms.append(name.replace("num__missingindicator_", "", 1))
        elif name.startswith("num__"):
            raw_features_for_terms.append(name.replace("num__", "", 1))
        elif name.startswith("cat__"):
            payload = name.replace("cat__", "", 1)
            mapped = name
            for column in sorted(categorical_columns, key=len, reverse=True):
                if payload == column or payload.startswith(f"{column}_"):
                    mapped = column
                    break
            raw_features_for_terms.append(mapped)
        else:
            raw_features_for_terms.append(name)

    std = design.std(axis=0)
    keep = std > 1e-12
    design = design[:, keep]
    selected_encoded_features = selected_encoded_features[keep]
    selected_coefficients = selected_coefficients[keep]
    raw_features_for_terms = [raw for raw, keep_value in zip(raw_features_for_terms, keep) if keep_value]
    std = std[keep]
    design = (design - design.mean(axis=0)) / std

    vif_rows = []
    for idx, encoded_feature in enumerate(selected_encoded_features):
        target_column = design[:, idx]
        other_columns = np.delete(design, idx, axis=1)
        if other_columns.shape[1] == 0:
            r_squared = 0.0
            vif = 1.0
            term_status = "single_active_term"
        else:
            regression_matrix = np.column_stack([np.ones(other_columns.shape[0]), other_columns])
            beta, *_ = np.linalg.lstsq(regression_matrix, target_column, rcond=None)
            fitted_values = regression_matrix @ beta
            ss_res = float(np.sum((target_column - fitted_values) ** 2))
            ss_tot = float(np.sum((target_column - target_column.mean()) ** 2))
            r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
            if pd.isna(r_squared):
                vif = np.nan
                term_status = "undefined"
            elif r_squared >= 1.0 - 1e-10:
                vif = np.inf
                term_status = "near_exact_collinearity"
            else:
                vif = 1.0 / (1.0 - r_squared)
                term_status = "ok"
        vif_rows.append(
            {
                "feature": encoded_feature,
                "raw_feature": raw_features_for_terms[idx],
                "coefficient": selected_coefficients[idx],
                "active_in_model": bool(abs(selected_coefficients[idx]) > 1e-12),
                "vif": vif,
                "r_squared": r_squared,
                "status": term_status,
            }
        )

    vif_table = pd.DataFrame(vif_rows)
    if vif_table.empty:
        break
    finite_vif = vif_table.copy()
    finite_vif["vif_for_mean"] = finite_vif["vif"].replace([np.inf, -np.inf], np.nan)
    finite_vif["status_rank"] = finite_vif["status"].map(status_rank).fillna(1)
    status_summary = finite_vif.sort_values(["raw_feature", "status_rank"], ascending=[True, False]).drop_duplicates("raw_feature")[["raw_feature", "status"]]
    finite_vif["active_in_model_int"] = finite_vif["active_in_model"].astype(bool).astype(int)
    finite_vif["near_exact_collinearity_int"] = (finite_vif["status"] == "near_exact_collinearity").astype(int)
    finite_vif["vif_gt_5_int"] = (finite_vif["vif"] > 5).astype(int)
    finite_vif["vif_gt_10_int"] = (finite_vif["vif"] > 10).astype(int)
    raw_vif = (
        finite_vif.groupby("raw_feature", as_index=False)
        .agg(
            **{
                "Encoded terms": ("feature", "count"),
                "Active encoded terms": ("active_in_model_int", "sum"),
                "Max VIF": ("vif", "max"),
                "Mean finite VIF": ("vif_for_mean", "mean"),
                "Near-exact collinearity terms": ("near_exact_collinearity_int", "sum"),
                "Terms with VIF > 5": ("vif_gt_5_int", "sum"),
                "Terms with VIF > 10": ("vif_gt_10_int", "sum"),
            }
        )
        .merge(status_summary, on="raw_feature", how="left")
        .rename(columns={"raw_feature": "Variable", "status": "Worst status"})
        .sort_values(["Max VIF", "Variable"], ascending=[False, True])
        .reset_index(drop=True)
    )
    raw_vif = raw_vif[raw_vif["Variable"].isin(selected_features)].copy()
    high_vif = raw_vif[raw_vif["Max VIF"] > VIF_PRUNING_THRESHOLD].copy()
    if high_vif.empty:
        break
    high_vif["univariate_abs_gini_train"] = high_vif["Variable"].map(quality["univariate_abs_gini_train"].fillna(0.0))
    high_vif = high_vif.sort_values(["Max VIF", "univariate_abs_gini_train"], ascending=[False, True])
    drop_row = high_vif.iloc[0]
    dropped = drop_row["Variable"]
    selected_features.remove(dropped)
    vif_pruning_rows.append(
        {
            "step": step,
            "dropped_feature": dropped,
            "max_vif": float(drop_row["Max VIF"]),
            "univariate_abs_gini_train": float(drop_row["univariate_abs_gini_train"]),
            "threshold": VIF_PRUNING_THRESHOLD,
        }
    )

vif_pruning = pd.DataFrame(vif_pruning_rows)



### Flagi selekcji w raporcie zmiennych


In [ ]:
feature_report["used_before_correlation_pruning"] = feature_report["feature"].isin(pre_correlation_features)
feature_report["used_before_vif_pruning"] = feature_report["feature"].isin(pre_vif_features)
feature_report["used_in_model"] = feature_report["feature"].isin(selected_features)
feature_report["correlation_pruned"] = feature_report["used_before_correlation_pruning"] & ~feature_report["used_before_vif_pruning"]
feature_report["vif_pruned"] = feature_report["used_before_vif_pruning"] & ~feature_report["used_in_model"]

if not selected_features:
    raise ValueError("No features passed data-quality screening.")



### Zapis audytu selekcji


In [ ]:
sample.attrs["sample_audit"]["feature_selection_mode"] = FEATURE_SELECTION_MODE
sample.attrs["sample_audit"]["correlation_pruning_threshold"] = CORRELATION_PRUNING_THRESHOLD
sample.attrs["sample_audit"]["vif_pruning_threshold"] = VIF_PRUNING_THRESHOLD
sample.attrs["sample_audit"]["raw_candidate_feature_count"] = len(raw_features)
sample.attrs["sample_audit"]["model_feature_count"] = len(selected_features)
sample.attrs["sample_audit"]["logistic_c"] = LOGISTIC_C
sample.attrs["sample_audit"]["numeric_add_indicators"] = NUMERIC_ADD_INDICATORS
sample.attrs["sample_audit"]["onehot_drop"] = ONEHOT_DROP
pd.DataFrame([sample.attrs["sample_audit"]]).to_csv(OUTPUT_DIR / "sample_definition.csv", index=False)
pd.Series(raw_features, name="feature").to_csv(OUTPUT_DIR / "raw_candidate_features.csv", index=False)
pd.Series(selected_features, name="feature").to_csv(OUTPUT_DIR / "selected_features.csv", index=False)
feature_report.to_csv(OUTPUT_DIR / "feature_quality.csv", index=False)
correlation_pruning.to_csv(OUTPUT_DIR / "correlation_pruning.csv", index=False)
vif_pruning.to_csv(OUTPUT_DIR / "vif_pruning.csv", index=False)
X = X_all[selected_features]

len(raw_features), len(pre_vif_features), len(selected_features), vif_pruning



## Krok 3: analiza zmiennych i raport Variable_report.xlsx
Binning, Big_scorecard, Gini_vars, arkusze zmiennych z wykresami oraz plik CSV z opisem binów.


### Binning i statystyki per zmienna


In [ ]:
big_frames = []
gini_rows = []
variable_details = {}
variable_definitions = []
variable_logit_mappers = {}

for feature in X.columns:
    series = X[feature]
    train_mask = masks["train"]
    test_mask = masks["test"]
    y_train = y.loc[train_mask]
    y_test = y.loc[test_mask]
    is_numeric = pd.api.types.is_numeric_dtype(series)
    specs = []

    if is_numeric:
        numeric = pd.to_numeric(series.loc[train_mask], errors="coerce")
        non_missing = numeric.dropna()
        thresholds = []
        if non_missing.nunique() > 1 and y_train.loc[non_missing.index].nunique() > 1:
            min_leaf = max(20, int(np.ceil(non_missing.shape[0] * ANALYSIS_MIN_BIN_SHARE)))
            tree = DecisionTreeClassifier(max_leaf_nodes=ANALYSIS_MAX_BINS, min_samples_leaf=min_leaf, random_state=RANDOM_STATE)
            tree.fit(non_missing.to_frame(), y_train.loc[non_missing.index])
            thresholds = sorted({float(threshold) for threshold in tree.tree_.threshold if threshold != -2 and np.isfinite(threshold)})
        if not thresholds and non_missing.nunique() > 1:
            n_bins = min(ANALYSIS_MAX_BINS, int(non_missing.nunique()))
            quantiles = np.linspace(0, 1, n_bins + 1)[1:-1]
            thresholds = sorted({float(value) for value in np.nanquantile(non_missing, quantiles) if np.isfinite(value)})
        edges = [-np.inf] + thresholds + [np.inf]
        for idx, (left, right) in enumerate(zip(edges[:-1], edges[1:])):
            if left == -np.inf:
                condition = f"{feature} < {right:.3f}".rstrip("0").rstrip(".")
            elif right == np.inf:
                condition = f"{left:.3f} <= {feature}".rstrip("0").rstrip(".")
            else:
                left_label = f"{left:.3f}".rstrip("0").rstrip(".")
                right_label = f"{right:.3f}".rstrip("0").rstrip(".")
                condition = f"{left_label} <= {feature} < {right_label}"
            specs.append({"raw_group": f"bin_{idx}", "left": left, "right": right, "condition": condition})
        if numeric.isna().any():
            specs.append({"raw_group": "missing", "left": np.nan, "right": np.nan, "condition": f"{feature} = Missing"})

        def_groups_train = pd.Series(index=series.loc[train_mask].index, dtype="object")
        numeric_train = pd.to_numeric(series.loc[train_mask], errors="coerce")
        interval_specs = [spec for spec in specs if spec["raw_group"] != "missing"]
        for spec in interval_specs:
            def_groups_train.loc[numeric_train.ge(spec["left"]) & numeric_train.lt(spec["right"])] = spec["raw_group"]
        missing_specs = [spec for spec in specs if spec["raw_group"] == "missing"]
        fallback = interval_specs[0]["raw_group"] if interval_specs else "missing"
        def_groups_train.loc[numeric_train.isna()] = missing_specs[0]["raw_group"] if missing_specs else fallback
        train_raw_groups = def_groups_train.fillna(fallback)
    else:
        values_train = series.loc[train_mask].astype("object").where(series.loc[train_mask].notna(), "__MISSING__").astype(str)
        counts = values_train.value_counts(dropna=False)
        shares = counts / counts.sum()
        rare = set(shares[shares < ANALYSIS_MIN_BIN_SHARE].index)
        if counts.shape[0] > 20:
            rare.update(counts.iloc[20:].index)
        frequent = [value for value in counts.index if value not in rare]
        bad_rates = pd.DataFrame({"value": values_train, "target": y_train}).groupby("value")["target"].mean()
        frequent_scores = bad_rates.reindex(frequent).fillna(0.0).sort_values(ascending=False)
        frequent = frequent_scores.index.tolist()
        n_groups = min(ANALYSIS_MAX_BINS, max(1, len(frequent)))
        if frequent:
            for idx, chunk in enumerate(np.array_split(frequent, n_groups)):
                chunk_values = [str(value) for value in chunk if len(chunk)]
                if not chunk_values:
                    continue
                display_values = ["Missing" if value == "__MISSING__" else value for value in chunk_values]
                if len(display_values) > 12:
                    display_values = display_values[:12] + [f"... +{len(chunk_values) - 12} more"]
                specs.append({"raw_group": f"cat_{idx}", "values": set(chunk_values), "condition": ", ".join(display_values), "is_other": False})
        if rare:
            specs.append({"raw_group": "other", "values": set(str(value) for value in rare), "condition": "<OTHERS>", "is_other": True})
        if not specs:
            all_values = [str(value) for value in counts.index]
            display_values = ["Missing" if value == "__MISSING__" else value for value in all_values]
            specs.append({"raw_group": "cat_0", "values": set(all_values), "condition": ", ".join(display_values), "is_other": False})
        value_to_group = {}
        other_group = None
        for spec in specs:
            if spec.get("is_other"):
                other_group = spec["raw_group"]
            for value in spec["values"]:
                value_to_group[value] = spec["raw_group"]
        fallback = other_group or specs[0]["raw_group"]
        train_raw_groups = values_train.map(value_to_group).fillna(fallback)

    condition_map = {spec["raw_group"]: spec["condition"] for spec in specs}
    train_summary = pd.DataFrame({"raw_group": train_raw_groups, "target": y_train}).groupby("raw_group", dropna=False)["target"].agg(Bad="sum", All="count").reset_index()
    train_summary["Good"] = train_summary["All"] - train_summary["Bad"]
    train_summary["BR"] = train_summary["Bad"] / train_summary["All"]
    train_summary["Logit"] = np.log((train_summary["Bad"] + EPSILON) / (train_summary["Good"] + EPSILON))
    train_summary["Variable"] = feature
    train_summary["Condition"] = train_summary["raw_group"].map(condition_map)
    train_summary["Share"] = train_summary["All"] / train_summary["All"].sum()
    train_bad_total = max(float(train_summary["Bad"].sum()), EPSILON)
    train_good_total = max(float(train_summary["Good"].sum()), EPSILON)
    train_summary["Bad share"] = train_summary["Bad"] / train_bad_total
    train_summary["Good share"] = train_summary["Good"] / train_good_total
    train_summary["Infomration Value"] = (train_summary["Good share"] - train_summary["Bad share"]) * np.log((train_summary["Good share"] + EPSILON) / (train_summary["Bad share"] + EPSILON))
    train_summary = train_summary.sort_values("BR", ascending=False).reset_index(drop=True)
    train_summary["GRP"] = train_summary.index
    train_summary["Type"] = "INT" if is_numeric else "NOM"
    group_map = dict(zip(train_summary["raw_group"], train_summary["GRP"]))

    if is_numeric:
        numeric_test = pd.to_numeric(series.loc[test_mask], errors="coerce")
        test_raw_groups = pd.Series(index=series.loc[test_mask].index, dtype="object")
        interval_specs = [spec for spec in specs if spec["raw_group"] != "missing"]
        for spec in interval_specs:
            test_raw_groups.loc[numeric_test.ge(spec["left"]) & numeric_test.lt(spec["right"])] = spec["raw_group"]
        missing_specs = [spec for spec in specs if spec["raw_group"] == "missing"]
        fallback = interval_specs[0]["raw_group"] if interval_specs else "missing"
        test_raw_groups.loc[numeric_test.isna()] = missing_specs[0]["raw_group"] if missing_specs else fallback
        test_raw_groups = test_raw_groups.fillna(fallback)
    else:
        values_test = series.loc[test_mask].astype("object").where(series.loc[test_mask].notna(), "__MISSING__").astype(str)
        test_raw_groups = values_test.map(value_to_group).fillna(fallback)

    test_groups = test_raw_groups.map(group_map).fillna(0).astype(int)
    test_summary = pd.DataFrame({"GRP": test_groups, "target": y_test}).groupby("GRP")["target"].agg(Bad_test="sum", All_test="count").reindex(train_summary["GRP"], fill_value=0).reset_index()
    test_bad_total = max(float(test_summary["Bad_test"].sum()), EPSILON)
    test_summary["Share test"] = test_summary["All_test"] / max(float(test_summary["All_test"].sum()), EPSILON)
    test_summary["Bad share test"] = test_summary["Bad_test"] / test_bad_total

    big = train_summary.merge(test_summary, on="GRP", how="left")
    big["Population Stability Index"] = (big["Share"] - big["Share test"]) * np.log((big["Share"] + EPSILON) / (big["Share test"] + EPSILON))
    big["Population Stability Index for bads"] = (big["Bad share"] - big["Bad share test"]) * np.log((big["Bad share"] + EPSILON) / (big["Bad share test"] + EPSILON))

    logit_by_group = big.set_index("GRP")["Logit"]
    train_groups = train_raw_groups.map(group_map).fillna(0).astype(int)
    train_scores = train_groups.map(logit_by_group)
    test_scores = test_groups.map(logit_by_group)
    gini_train = 0.0 if pd.Series(train_scores).nunique(dropna=True) < 2 else float(abs(2.0 * roc_auc_score(y_train, train_scores) - 1.0))
    gini_test = 0.0 if pd.Series(test_scores).nunique(dropna=True) < 2 else float(abs(2.0 * roc_auc_score(y_test, test_scores) - 1.0))
    relative_gini = abs(gini_train - gini_test) / gini_train if gini_train > 0 else 0.0
    train_series = series.loc[train_mask]
    non_missing = train_series.dropna()
    mode = non_missing.mode().iloc[0] if not non_missing.empty else np.nan
    mode_share = float((non_missing == mode).mean()) if not non_missing.empty else np.nan

    gini_rows.append(
        {
            "Variable": feature,
            "Gini train": gini_train,
            "Gini test": gini_test,
            "R. Gini": relative_gini,
            "Infomration Value": float(big["Infomration Value"].sum()),
            "Population Stability Index": float(big["Population Stability Index"].sum()),
            "Population Stability Index for bads": float(big["Population Stability Index for bads"].sum()),
            "Missing percent": float(train_series.isna().mean()),
            "Number of distinct": int(train_series.nunique(dropna=True)),
            "Mode": mode,
            "P. mode": mode_share,
            "Type": "INT" if is_numeric else "NOM",
        }
    )

    if is_numeric:
        numeric_all = pd.to_numeric(series, errors="coerce")
        all_raw_groups = pd.Series(index=series.index, dtype="object")
        interval_specs = [spec for spec in specs if spec["raw_group"] != "missing"]
        for spec in interval_specs:
            all_raw_groups.loc[numeric_all.ge(spec["left"]) & numeric_all.lt(spec["right"])] = spec["raw_group"]
        missing_specs = [spec for spec in specs if spec["raw_group"] == "missing"]
        fallback = interval_specs[0]["raw_group"] if interval_specs else "missing"
        all_raw_groups.loc[numeric_all.isna()] = missing_specs[0]["raw_group"] if missing_specs else fallback
        all_raw_groups = all_raw_groups.fillna(fallback)
    else:
        values_all = series.astype("object").where(series.notna(), "__MISSING__").astype(str)
        all_raw_groups = values_all.map(value_to_group).fillna(fallback)

    all_groups = all_raw_groups.map(group_map).fillna(0).astype(int)
    years = sorted(sample[PERIOD_COLUMN].astype(str).str[:4].unique())
    group_ids = big["GRP"].tolist()
    time_frame = pd.DataFrame({"GRP": all_groups, "Time": sample[PERIOD_COLUMN].astype(str).str[:4], "target": y})
    yearly_totals = time_frame.groupby("Time")["target"].count().rename("Year_All")
    time_summary = time_frame.groupby(["GRP", "Time"])["target"].agg(Bad="sum", All="count").reindex(pd.MultiIndex.from_product([group_ids, years], names=["GRP", "Time"]), fill_value=0).reset_index()
    time_summary = time_summary.merge(yearly_totals, on="Time", how="left")
    time_summary["Good"] = time_summary["All"] - time_summary["Bad"]
    time_summary["BR"] = np.where(time_summary["All"] > 0, time_summary["Bad"] / time_summary["All"], np.nan)
    time_summary["Share"] = time_summary["All"] / time_summary["Year_All"]
    time_summary = time_summary[["GRP", "Time", "Bad", "All", "Good", "BR", "Share"]]

    detail = big[["GRP", "Condition", "BR", "Share", "All", "Bad", "Good", "Logit"]].copy()
    big = big[["Variable", "Condition", "BR", "Share", "All", "Bad", "Good", "Logit", "GRP", "Type", "Bad share", "Good share", "Infomration Value", "Share test", "Bad share test", "Population Stability Index", "Population Stability Index for bads"]]
    variable_logit_mappers[feature] = {
        "type": "INT" if is_numeric else "NOM",
        "specs": specs,
        "group_map": group_map,
        "logit_by_group": logit_by_group,
    }
    big_frames.append(big)
    variable_details[feature] = (detail, time_summary)
    for _, row in detail.iterrows():
        variable_definitions.append({"Variable": feature, "GRP": row["GRP"], "Definition": row["Condition"], "Bad_rate": row["BR"], "Share": row["Share"], "All": row["All"], "Bad": row["Bad"], "Good": row["Good"]})



### Złożenie Big_scorecard i Gini_vars


In [ ]:
big_scorecard = pd.concat(big_frames, ignore_index=True)
gini_vars = pd.DataFrame(gini_rows).sort_values("Gini train", ascending=False).reset_index(drop=True)



### Zapis Big_scorecard.xlsx i Gini_vars.xlsx


In [ ]:
for path, dataframe in [(OUTPUT_DIR / "Big_scorecard.xlsx", big_scorecard), (OUTPUT_DIR / "Gini_vars.xlsx", gini_vars)]:
    with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
        dataframe.to_excel(writer, sheet_name="Sheet1", index=False)
        workbook = writer.book
        worksheet = writer.sheets["Sheet1"]
        header_format = workbook.add_format({"bold": True, "bg_color": "#D9EAF7"})
        percent_format = workbook.add_format({"num_format": "0.0%"})
        numeric_format = workbook.add_format({"num_format": "#,##0.000"})
        integer_format = workbook.add_format({"num_format": "#,##0"})
        percent_tokens = ("BR", "Share", "Gini", "PSI", "percent", "P. mode")
        integer_columns = {"All", "Bad", "Good", "GRP", "Number of distinct"}
        for col_idx, column in enumerate(dataframe.columns):
            worksheet.write(0, col_idx, column, header_format)
            width = max(len(str(column)) + 2, 10)
            if not dataframe.empty:
                width = max(width, min(45, int(dataframe[column].astype(str).str.len().quantile(0.95)) + 2))
            cell_format = None
            if any(token in str(column) for token in percent_tokens):
                cell_format = percent_format
            elif column in integer_columns or str(column).endswith("_test"):
                cell_format = integer_format
            elif pd.api.types.is_numeric_dtype(dataframe[column]):
                cell_format = numeric_format
            worksheet.set_column(col_idx, col_idx, width, cell_format)
        if dataframe.shape[0] > 0 and dataframe.shape[1] > 0:
            worksheet.autofilter(0, 0, dataframe.shape[0], dataframe.shape[1] - 1)
        worksheet.freeze_panes(1, 0)



### Zapis variable_definitions.csv


In [ ]:
pd.DataFrame(variable_definitions).to_csv(OUTPUT_DIR / "variable_definitions.csv", index=False)



### Wybór zmiennych do Variable_report.xlsx


In [ ]:
report_vars = gini_vars.loc[
    (gini_vars["Gini train"] > ASB_GINI_TRAIN_MIN)
    & (gini_vars["R. Gini"] < ASB_RELATIVE_GINI_MAX)
    & (gini_vars["Population Stability Index"] < ASB_PSI_MAX)
    & (gini_vars["Population Stability Index for bads"] < ASB_PSI_BAD_MAX)
]["Variable"].tolist()
if len(report_vars) < 1:
    report_vars = gini_vars.head(MAX_VARIABLE_REPORT_SHEETS)["Variable"].tolist()
else:
    report_vars = report_vars[:MAX_VARIABLE_REPORT_SHEETS]



### Zapis Variable_report.xlsx z wykresami


In [ ]:
used_sheet_names = set()
with pd.ExcelWriter(OUTPUT_DIR / "Variable_report.xlsx", engine="xlsxwriter") as writer:
    gini_vars.to_excel(writer, sheet_name="Variable", startrow=1, index=False)
    workbook = writer.book
    worksheet = writer.sheets["Variable"]
    header_format = workbook.add_format({"bold": True, "bg_color": "#D9EAF7"})
    percent_format = workbook.add_format({"num_format": "0.0%"})
    numeric_format = workbook.add_format({"num_format": "#,##0.000"})
    integer_format = workbook.add_format({"num_format": "#,##0"})
    percent_tokens = ("BR", "Share", "Gini", "PSI", "percent", "P. mode")
    integer_columns = {"All", "Bad", "Good", "GRP", "Number of distinct"}
    for col_idx, column in enumerate(gini_vars.columns):
        worksheet.write(1, col_idx, column, header_format)
        width = max(len(str(column)) + 2, 10)
        if not gini_vars.empty:
            width = max(width, min(45, int(gini_vars[column].astype(str).str.len().quantile(0.95)) + 2))
        cell_format = percent_format if any(token in str(column) for token in percent_tokens) else numeric_format if pd.api.types.is_numeric_dtype(gini_vars[column]) else None
        worksheet.set_column(col_idx, col_idx, width, cell_format)
    worksheet.freeze_panes(2, 0)
    worksheet.autofilter(1, 0, gini_vars.shape[0] + 1, gini_vars.shape[1] - 1)
    used_sheet_names.add("Variable")
    title_format = workbook.add_format({"bold": True, "size": 18})

    for feature in report_vars:
        detail, time_summary = variable_details[feature]
        invalid = set("[]:*?/\\")
        clean = "".join("_" if char in invalid else char for char in str(feature))
        clean = clean[:31] or "Sheet"
        sheet_name = clean
        counter = 1
        while sheet_name in used_sheet_names:
            suffix = f"_{counter}"
            sheet_name = clean[: 31 - len(suffix)] + suffix
            counter += 1
        used_sheet_names.add(sheet_name)
        detail.to_excel(writer, sheet_name=sheet_name, startrow=2, index=False)
        time_summary.to_excel(writer, sheet_name=sheet_name, startrow=2, startcol=8, index=False)
        worksheet = writer.sheets[sheet_name]
        worksheet.write(0, 0, f"Variable: {feature}", title_format)
        for col_idx, column in enumerate(detail.columns):
            worksheet.write(2, col_idx, column, header_format)
            width = max(len(str(column)) + 2, 10)
            if not detail.empty:
                width = max(width, min(45, int(detail[column].astype(str).str.len().quantile(0.95)) + 2))
            cell_format = percent_format if any(token in str(column) for token in percent_tokens) else integer_format if column in integer_columns else numeric_format if pd.api.types.is_numeric_dtype(detail[column]) else None
            worksheet.set_column(col_idx, col_idx, width, cell_format)
        for col_idx, column in enumerate(time_summary.columns):
            worksheet.write(2, 8 + col_idx, column, header_format)
            width = max(len(str(column)) + 2, 10)
            if not time_summary.empty:
                width = max(width, min(45, int(time_summary[column].astype(str).str.len().quantile(0.95)) + 2))
            cell_format = percent_format if any(token in str(column) for token in percent_tokens) else integer_format if column in integer_columns else numeric_format if pd.api.types.is_numeric_dtype(time_summary[column]) else None
            worksheet.set_column(8 + col_idx, 8 + col_idx, width, cell_format)
        worksheet.freeze_panes(3, 0)
        detail_rows = detail.shape[0]
        time_rows = time_summary.shape[0]
        if detail_rows and time_rows:
            first_data_row = 4
            groups = detail_rows
            years_per_group = int(time_rows / groups) if groups else 0
            if years_per_group:
                for metric_name, column_letter, anchor_offset in [("BR", "N", 6), ("Share", "O", 22)]:
                    chart = workbook.add_chart({"type": "line"})
                    chart.set_title({"name": metric_name})
                    chart.set_x_axis({"name": "Time", "num_font": {"rotation": 45}})
                    chart.set_legend({"position": "bottom"})
                    for idx in range(groups):
                        start = first_data_row + idx * years_per_group
                        end = start + years_per_group - 1
                        chart.add_series({"name": f"='{sheet_name}'!$I${start}", "categories": f"='{sheet_name}'!$J${first_data_row}:$J${first_data_row + years_per_group - 1}", "values": f"='{sheet_name}'!${column_letter}${start}:${column_letter}${end}"})
                    worksheet.insert_chart(f"A{detail_rows + anchor_offset}", chart)

gini_vars.head()



## Krok 4: trening modelu i metryki
Pipeline sklearn, regresja logistyczna L1, scoring oraz metryki dla train/validation/test.


### Budowa i trening pipeline sklearn


In [ ]:
numeric_columns = X.select_dtypes(include="number").columns.tolist()
categorical_columns = [column for column in X.columns if column not in numeric_columns]
numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=NUMERIC_ADD_INDICATORS)),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, drop=ONEHOT_DROP)),
    ]
)
preprocess = ColumnTransformer(transformers=[("num", numeric_pipe, numeric_columns), ("cat", categorical_pipe, categorical_columns)])
classifier = LogisticRegression(penalty="l1", solver="liblinear", C=LOGISTIC_C, max_iter=2000, random_state=RANDOM_STATE)
model = Pipeline(steps=[("preprocess", preprocess), ("model", classifier)])
model.fit(X.loc[masks["train"]], y.loc[masks["train"]])



### Scoring i metryki train / validation / test


In [ ]:
metrics = []
probabilities = {}
scores = {}
for split_name, mask in masks.items():
    scores[split_name] = model.decision_function(X.loc[mask])
    probabilities[split_name] = model.predict_proba(X.loc[mask])[:, 1]
    y_true = y.loc[mask]
    probability = probabilities[split_name]
    metrics.append(
        {
            "model_id": MODEL_ID,
            "split": split_name,
            "n_obs": int(len(y_true)),
            "bad_rate": float(y_true.mean()),
            "predicted_pd_mean": float(np.mean(probability)),
            "auc": float(roc_auc_score(y_true, probability)),
            "gini": float(2.0 * roc_auc_score(y_true, probability) - 1.0),
            "brier": float(brier_score_loss(y_true, probability)),
            "log_loss": float(log_loss(y_true, probability)),
        }
    )
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
metrics_df



## Krok 5: walidacja po produkcie i submodele
Metryki osobno dla produktów oraz oddzielne modele produktowe do porównania.


### Walidacja modelu po produkcie


In [ ]:
product_rows = []
for split_name, mask in masks.items():
    frame = sample.loc[mask, ["product"]].copy()
    frame["target"] = y.loc[mask].values
    frame["probability"] = probabilities[split_name]
    for product, group in frame.groupby("product", dropna=False):
        if group["target"].nunique() < 2 or group["probability"].nunique() < 2:
            auc = np.nan
            gini = np.nan
            brier = np.nan
            ll = np.nan
            status = "insufficient_classes"
        else:
            auc = float(roc_auc_score(group["target"], group["probability"]))
            gini = float(2.0 * auc - 1.0)
            brier = float(brier_score_loss(group["target"], group["probability"]))
            ll = float(log_loss(group["target"], group["probability"]))
            status = "ok"
        product_rows.append(
            {
                "model_id": MODEL_ID,
                "split": split_name,
                "product": product,
                "n_obs": int(group.shape[0]),
                "bad_rate": float(group["target"].mean()),
                "predicted_pd_mean": float(group["probability"].mean()),
                "auc": auc,
                "gini": gini,
                "brier": brier,
                "log_loss": ll,
                "status": status,
            }
        )
product_metrics = pd.DataFrame(product_rows)
product_metrics.to_csv(OUTPUT_DIR / "metrics_by_product.csv", index=False)



### Trening submodeli produktowych


In [ ]:
submodel_rows = []
for product in sorted(sample["product"].dropna().unique()):
    product_mask = sample["product"].eq(product)
    sub_sample = sample.loc[product_mask].reset_index(drop=True)
    sub_X = X.loc[product_mask].reset_index(drop=True)
    sub_y = y.loc[product_mask].reset_index(drop=True)
    try:
        sub_periods = sub_sample[[PERIOD_COLUMN]].copy()
        sub_periods["row_number"] = np.arange(sub_sample.shape[0])
        sub_period_counts = sub_periods.groupby(PERIOD_COLUMN).size().sort_index().rename("n_obs").reset_index()
        sub_test_target = max(MIN_HOLDOUT_ROWS, int(round(sub_sample.shape[0] * TEST_FRACTION)))
        sub_validation_target = max(MIN_HOLDOUT_ROWS, int(round(sub_sample.shape[0] * VALIDATION_FRACTION)))
        sub_period_counts["cum_from_end"] = sub_period_counts["n_obs"][::-1].cumsum()[::-1]
        sub_test_periods = sub_period_counts.loc[sub_period_counts["cum_from_end"] <= sub_test_target, PERIOD_COLUMN]
        if sub_test_periods.empty:
            sub_test_periods = sub_period_counts.tail(1)[PERIOD_COLUMN]
        sub_test_start = sub_test_periods.min()
        sub_train_validation_periods = sub_period_counts.loc[sub_period_counts[PERIOD_COLUMN] < sub_test_start].copy()
        sub_train_validation_periods["cum_from_end"] = sub_train_validation_periods["n_obs"][::-1].cumsum()[::-1]
        sub_validation_periods = sub_train_validation_periods.loc[sub_train_validation_periods["cum_from_end"] <= sub_validation_target, PERIOD_COLUMN]
        if sub_validation_periods.empty:
            sub_validation_periods = sub_train_validation_periods.tail(1)[PERIOD_COLUMN]
        sub_validation_start = sub_validation_periods.min()
        sub_masks = {
            "train": sub_sample[PERIOD_COLUMN] < sub_validation_start,
            "validation": sub_sample[PERIOD_COLUMN].between(sub_validation_start, sub_test_start, inclusive="left"),
            "test": sub_sample[PERIOD_COLUMN] >= sub_test_start,
        }
        if any(int(mask.sum()) == 0 for mask in sub_masks.values()):
            raise ValueError("Temporal split produced an empty train, validation, or test sample.")
        for sub_split_name, sub_mask in sub_masks.items():
            if sub_y.loc[sub_mask].nunique() < 2:
                raise ValueError(f"Split {sub_split_name} has only one target class.")
    except ValueError as error:
        submodel_rows.append(
            {
                "model_id": MODEL_ID,
                "submodel_product": product,
                "split": "all",
                "n_obs": int(sub_sample.shape[0]),
                "bad_rate": float(sub_y.mean()) if len(sub_y) else np.nan,
                "predicted_pd_mean": np.nan,
                "auc": np.nan,
                "gini": np.nan,
                "brier": np.nan,
                "log_loss": np.nan,
                "status": f"skipped: {error}",
            }
        )
        continue

    sub_numeric_columns = sub_X.select_dtypes(include="number").columns.tolist()
    sub_categorical_columns = [column for column in sub_X.columns if column not in sub_numeric_columns]
    sub_numeric_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median", add_indicator=NUMERIC_ADD_INDICATORS)), ("scaler", StandardScaler())])
    sub_categorical_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, drop=ONEHOT_DROP))])
    sub_preprocess = ColumnTransformer(transformers=[("num", sub_numeric_pipe, sub_numeric_columns), ("cat", sub_categorical_pipe, sub_categorical_columns)])
    sub_classifier = LogisticRegression(penalty="l1", solver="liblinear", C=LOGISTIC_C, max_iter=2000, random_state=RANDOM_STATE)
    sub_model = Pipeline(steps=[("preprocess", sub_preprocess), ("model", sub_classifier)])
    sub_model.fit(sub_X.loc[sub_masks["train"]], sub_y.loc[sub_masks["train"]])

    for split_name, mask in sub_masks.items():
        probability = sub_model.predict_proba(sub_X.loc[mask])[:, 1]
        y_true = sub_y.loc[mask]
        if len(y_true) == 0 or y_true.nunique() < 2 or pd.Series(probability).nunique() < 2:
            auc = np.nan
            gini = np.nan
            brier = np.nan
            ll = np.nan
            status = "insufficient_classes"
        else:
            auc = float(roc_auc_score(y_true, probability))
            gini = float(2.0 * auc - 1.0)
            brier = float(brier_score_loss(y_true, probability))
            ll = float(log_loss(y_true, probability))
            status = "ok"
        submodel_rows.append(
            {
                "model_id": MODEL_ID,
                "split": split_name,
                "n_obs": int(len(y_true)),
                "bad_rate": float(y_true.mean()) if len(y_true) else np.nan,
                "predicted_pd_mean": float(np.mean(probability)) if len(probability) else np.nan,
                "auc": auc,
                "gini": gini,
                "brier": brier,
                "log_loss": ll,
                "submodel_product": product,
                "validation_start": sub_validation_start,
                "test_start": sub_test_start,
                "status": status,
            }
        )

submodel_metrics = pd.DataFrame(submodel_rows)
submodel_metrics.to_csv(OUTPUT_DIR / "submodel_metrics.csv", index=False)
product_metrics, submodel_metrics



## Krok 6: predykcje, raport modelu, segmenty i kod SAS
Zapis CSV, `gini_curves_model.xlsx`, `Model_report.xlsx`, `Segments3_report.xlsx` i `scoring_code.sas`.


### Zapis predykcji


In [ ]:
all_predictions = []
for split_name, mask in masks.items():
    predictions = sample.loc[mask, [ID_COLUMN, PERIOD_COLUMN, "product", "decision", TARGET]].copy()
    predictions[SCORE_COLUMN] = scores[split_name]
    predictions[PD_COLUMN] = probabilities[split_name]
    predictions["split"] = split_name
    all_predictions.append(predictions)
pd.concat(all_predictions, ignore_index=True).to_csv(OUTPUT_DIR / "predictions.csv", index=False)
all_predictions[-1].to_csv(OUTPUT_DIR / "oot_test_predictions.csv", index=False)



### Kalibracja testowa


In [ ]:
scored_test_cal = pd.DataFrame({"target": y.loc[masks["test"]], PD_COLUMN: probabilities["test"]})
scored_test_cal["bucket"] = pd.qcut(scored_test_cal[PD_COLUMN].rank(method="first"), q=10, labels=False, duplicates="drop")
test_calibration = scored_test_cal.groupby("bucket").agg(n_obs=("target", "size"), bads=("target", "sum"), observed_default_rate=("target", "mean"), mean_pd=(PD_COLUMN, "mean"), min_pd=(PD_COLUMN, "min"), max_pd=(PD_COLUMN, "max")).reset_index()
test_calibration["calibration_error"] = test_calibration["mean_pd"] - test_calibration["observed_default_rate"]
test_calibration.to_csv(OUTPUT_DIR / "oot_test_calibration.csv", index=False)



### Istotność przekształconych cech


In [ ]:
preprocess = model.named_steps["preprocess"]
classifier = model.named_steps["model"]
importance = pd.DataFrame({"feature": preprocess.get_feature_names_out(), "coefficient": classifier.coef_[0], "importance": np.abs(classifier.coef_[0])}).sort_values("importance", ascending=False).reset_index(drop=True)
importance.head(100).to_csv(OUTPUT_DIR / "top_feature_importance.csv", index=False)



### Gini curves workbook


In [ ]:
scored_for_gini = pd.DataFrame({TARGET: y})
scored_for_gini[SCORE_COLUMN] = pd.Series(dtype=float, index=sample.index)
for split_name, mask in masks.items():
    scored_for_gini.loc[mask, SCORE_COLUMN] = scores[split_name]
scored_for_gini["rank"] = scored_for_gini[SCORE_COLUMN].rank()
scored_for_gini["grouping"] = round(scored_for_gini["rank"] * (20 - 1) / (len(scored_for_gini[SCORE_COLUMN]) + 1))
sss = pd.DataFrame(scored_for_gini.groupby(["grouping"]).agg({TARGET: ["sum", "count"]}))
sss = pd.DataFrame(sss[TARGET]).reset_index()
sss["count"] = sss["count"] - sss["sum"]
sss = sss.rename(columns={"count": "goods", "sum": "bads"})[["bads", "goods"]]
gini_curves_workbook = load_workbook(GINI_CURVES_TEMPLATE_PATH)
gini_curves_sheet = gini_curves_workbook.active
for i in range(sss.shape[0]):
    gini_curves_sheet.cell(row=9 + i, column=4).value = sss["bads"][i]
    gini_curves_sheet.cell(row=9 + i, column=5).value = sss["goods"][i]
gini_curves_workbook.save(GINI_CURVES_OUTPUT_PATH)



### Pełne score i PD


In [ ]:
full_probability = pd.Series(index=sample.index, dtype=float)
full_score = pd.Series(index=sample.index, dtype=float)
for split_name, mask in masks.items():
    full_probability.loc[mask] = probabilities[split_name]
    full_score.loc[mask] = scores[split_name]



### Istotność surowych zmiennych


In [ ]:
numeric_columns = list(preprocess.transformers_[0][2])
categorical_columns = list(preprocess.transformers_[1][2])
importance["raw_feature"] = None
for row_idx, name in importance["feature"].items():
    if name.startswith("num__missingindicator_"):
        mapped = name.replace("num__missingindicator_", "", 1)
    elif name.startswith("num__"):
        mapped = name.replace("num__", "", 1)
    elif name.startswith("cat__"):
        payload = name.replace("cat__", "", 1)
        mapped = name
        for column in sorted(categorical_columns, key=len, reverse=True):
            if payload == column or payload.startswith(f"{column}_"):
                mapped = column
                break
    else:
        mapped = name
    importance.loc[row_idx, "raw_feature"] = mapped
importance["nonzero_term"] = (importance["importance"] > 1e-12).astype(int)
raw_importance = importance.groupby("raw_feature", as_index=False).agg(coefficient_abs_sum=("importance", "sum"), max_abs_coefficient=("importance", "max"), nonzero_terms=("nonzero_term", "sum")).rename(columns={"raw_feature": "Variable"}).sort_values("coefficient_abs_sum", ascending=False).reset_index(drop=True)
total_importance = raw_importance["coefficient_abs_sum"].sum()
raw_importance["Importance"] = raw_importance["coefficient_abs_sum"] / total_importance if total_importance else 0.0



### VIF finalnego modelu


In [ ]:
feature_names = np.array(preprocess.get_feature_names_out())
coefficients = classifier.coef_[0]
design = preprocess.transform(X.loc[masks["train"]])
if hasattr(design, "toarray"):
    design = design.toarray()
design = np.asarray(design, dtype=float)
raw_features_for_terms = []
for name in feature_names:
    if name.startswith("num__missingindicator_"):
        raw_features_for_terms.append(name.replace("num__missingindicator_", "", 1))
    elif name.startswith("num__"):
        raw_features_for_terms.append(name.replace("num__", "", 1))
    elif name.startswith("cat__"):
        payload = name.replace("cat__", "", 1)
        mapped = name
        for column in sorted(categorical_columns, key=len, reverse=True):
            if payload == column or payload.startswith(f"{column}_"):
                mapped = column
                break
        raw_features_for_terms.append(mapped)
    else:
        raw_features_for_terms.append(name)
std = design.std(axis=0)
keep = std > 1e-12
design = design[:, keep]
selected_encoded_features = feature_names[keep]
selected_coefficients = coefficients[keep]
raw_features_for_terms = [raw for raw, keep_value in zip(raw_features_for_terms, keep) if keep_value]
std = std[keep]
design = (design - design.mean(axis=0)) / std
vif_rows = []
for idx, encoded_feature in enumerate(selected_encoded_features):
    target_column = design[:, idx]
    other_columns = np.delete(design, idx, axis=1)
    if other_columns.shape[1] == 0:
        r_squared = 0.0
        vif = 1.0
        term_status = "single_active_term"
    else:
        regression_matrix = np.column_stack([np.ones(other_columns.shape[0]), other_columns])
        beta, *_ = np.linalg.lstsq(regression_matrix, target_column, rcond=None)
        fitted_values = regression_matrix @ beta
        ss_res = float(np.sum((target_column - fitted_values) ** 2))
        ss_tot = float(np.sum((target_column - target_column.mean()) ** 2))
        r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
        if pd.isna(r_squared):
            vif = np.nan
            term_status = "undefined"
        elif r_squared >= 1.0 - 1e-10:
            vif = np.inf
            term_status = "near_exact_collinearity"
        else:
            vif = 1.0 / (1.0 - r_squared)
            term_status = "ok"
    vif_rows.append({"feature": encoded_feature, "raw_feature": raw_features_for_terms[idx], "coefficient": selected_coefficients[idx], "active_in_model": bool(abs(selected_coefficients[idx]) > 1e-12), "vif": vif, "r_squared": r_squared, "status": term_status})
vif_all = pd.DataFrame(vif_rows).sort_values(["vif", "feature"], ascending=[False, True]).reset_index(drop=True)
vif_active = vif_all.loc[vif_all["active_in_model"]].reset_index(drop=True)
finite_vif = vif_all.copy()
finite_vif["vif_for_mean"] = finite_vif["vif"].replace([np.inf, -np.inf], np.nan)
finite_vif["status_rank"] = finite_vif["status"].map({"ok": 0, "single_active_term": 0, "undefined": 1, "near_exact_collinearity": 2}).fillna(1)
status_summary = finite_vif.sort_values(["raw_feature", "status_rank"], ascending=[True, False]).drop_duplicates("raw_feature")[["raw_feature", "status"]]
finite_vif["active_in_model_int"] = finite_vif["active_in_model"].astype(bool).astype(int)
finite_vif["near_exact_collinearity_int"] = (finite_vif["status"] == "near_exact_collinearity").astype(int)
finite_vif["vif_gt_5_int"] = (finite_vif["vif"] > 5).astype(int)
finite_vif["vif_gt_10_int"] = (finite_vif["vif"] > 10).astype(int)
raw_vif = finite_vif.groupby("raw_feature", as_index=False).agg(**{"Encoded terms": ("feature", "count"), "Active encoded terms": ("active_in_model_int", "sum"), "Max VIF": ("vif", "max"), "Mean finite VIF": ("vif_for_mean", "mean"), "Near-exact collinearity terms": ("near_exact_collinearity_int", "sum"), "Terms with VIF > 5": ("vif_gt_5_int", "sum"), "Terms with VIF > 10": ("vif_gt_10_int", "sum")}).merge(status_summary, on="raw_feature", how="left").rename(columns={"raw_feature": "Variable", "status": "Worst status"}).sort_values(["Max VIF", "Variable"], ascending=[False, True]).reset_index(drop=True)
raw_importance = raw_importance.merge(raw_vif, on="Variable", how="left")
final_variables = raw_importance.loc[raw_importance["coefficient_abs_sum"] > 1e-12, "Variable"].tolist()
scorecard = big_scorecard[big_scorecard["Variable"].isin(final_variables)].copy()
model_gini_vars = gini_vars[gini_vars["Variable"].isin(final_variables)].copy()
metrics_by_split = {row["split"]: row for row in metrics}


def assign_raw_group(series, mapper):
    specs = mapper["specs"]
    if mapper["type"] == "INT":
        numeric = pd.to_numeric(series, errors="coerce")
        result = pd.Series(index=series.index, dtype="object")
        interval_specs = [spec for spec in specs if spec["raw_group"] != "missing"]
        for spec in interval_specs:
            result.loc[numeric.ge(spec["left"]) & numeric.lt(spec["right"])] = spec["raw_group"]
        missing_specs = [spec for spec in specs if spec["raw_group"] == "missing"]
        fallback = interval_specs[0]["raw_group"] if interval_specs else "missing"
        if missing_specs:
            result.loc[numeric.isna()] = missing_specs[0]["raw_group"]
        else:
            result.loc[numeric.isna()] = fallback
        return result.fillna(fallback)
    values = series.astype("object").where(series.notna(), "__MISSING__").astype(str)
    value_to_group = {}
    other_group = None
    for spec in specs:
        if spec.get("is_other"):
            other_group = spec["raw_group"]
        for value in spec.get("values", []):
            value_to_group[value] = spec["raw_group"]
    fallback = other_group or specs[0]["raw_group"]
    return values.map(value_to_group).fillna(fallback)


def build_logit_frame(frame, variables):
    logit_data = {}
    for feature in variables:
        mapper = variable_logit_mappers[feature]
        raw_groups = assign_raw_group(frame[feature], mapper)
        groups = raw_groups.map(mapper["group_map"]).fillna(0).astype(int)
        logit_data[feature] = groups.map(mapper["logit_by_group"])
    return pd.DataFrame(logit_data, index=frame.index)


logit_train = build_logit_frame(X.loc[masks["train"]], final_variables)
logit_design = sm.add_constant(logit_train[final_variables], has_constant="add")
logit_model = sm.Logit(y.loc[masks["train"]], logit_design).fit(disp=0, method="newton")
alpha = float(logit_model.params["const"])
betas = logit_model.params.drop("const")
coef = pd.DataFrame({"Variable": betas.index, "Beta": betas.values})
scorecard = scorecard.merge(coef, on="Variable", how="left")
factor = SCORECARD_FACTOR
v = len(final_variables)
alpp = scorecard["Logit"] * scorecard["Beta"] * factor
alp = -alpp.sum() + SCORECARD_BASE_POINTS
alpp2 = scorecard.loc[scorecard["GRP"] == 0, ["Variable", "Logit", "Beta"]].copy()
alpp2["FBeta"] = alpp2["Logit"] * alpp2["Beta"]
alpp2 = alpp2[["Variable", "FBeta"]]
scorecard = scorecard.merge(alpp2, on="Variable", how="left")
scorecard["Score"] = -(
    scorecard["Logit"] * scorecard["Beta"] - scorecard["FBeta"] + alpha / v
) * factor + alp / v
scorecard["Score"] = scorecard["Score"].round()
scorecard_scoring = scorecard.copy()
vars_sc = list(pd.unique(scorecard_scoring["Variable"]))
vars_sc_num = list(pd.unique(scorecard_scoring.loc[scorecard_scoring["Type"] == "INT", "Variable"]))
vars_sc_nom = list(pd.unique(scorecard_scoring.loc[scorecard_scoring["Type"] == "NOM", "Variable"]))



### Miary główne do Model_report.xlsx


In [ ]:
main_measures = pd.DataFrame(
    [
        ("Model ID", MODEL_ID),
        ("Target", TARGET),
        ("Application product", APPLICATION_PRODUCT),
        ("Rows", int(sample.shape[0])),
        ("Event rate", float(y.mean())),
        ("Gini train", metrics_by_split["train"]["gini"]),
        ("Gini validation", metrics_by_split["validation"]["gini"]),
        ("Gini test", metrics_by_split["test"]["gini"]),
        ("AUC train", metrics_by_split["train"]["auc"]),
        ("AUC validation", metrics_by_split["validation"]["auc"]),
        ("AUC test", metrics_by_split["test"]["auc"]),
        ("Test bad rate", metrics_by_split["test"]["bad_rate"]),
        ("Test mean PD", metrics_by_split["test"]["predicted_pd_mean"]),
        ("Selected features", int(gini_vars.shape[0])),
        ("Final non-zero raw features", len(final_variables)),
        ("Raw features with VIF", int(raw_vif.shape[0])),
        ("Encoded terms with VIF", int(vif_all.shape[0])),
        ("Max all-feature VIF", float(vif_all["vif"].max())),
        ("All-feature terms with VIF > 5", int((vif_all["vif"] > 5).sum())),
        ("All-feature terms with VIF > 10", int((vif_all["vif"] > 10).sum())),
        ("Max active-term VIF", float(vif_active["vif"].max())),
        ("Active terms with VIF > 5", int((vif_active["vif"] > 5).sum())),
        ("Active terms with VIF > 10", int((vif_active["vif"] > 10).sum())),
    ],
    columns=["Measure", "Value"],
)



### Gini over time do Model_report.xlsx


In [ ]:
gini_time_rows = []
gini_time_frame = pd.DataFrame({"Time": sample[PERIOD_COLUMN].astype(str).str[:4], "target": y, "probability": full_probability})
for period, group in gini_time_frame.groupby("Time"):
    if group["target"].nunique() < 2:
        period_gini = np.nan
    else:
        period_gini = 2.0 * roc_auc_score(group["target"], group["probability"]) - 1.0
    gini_time_rows.append({"Time": period, "Gini": period_gini, "n_obs": int(group.shape[0])})
gini_over_time_df = pd.DataFrame(gini_time_rows)



### Zapis Model_report.xlsx z wykresami


In [ ]:
with pd.ExcelWriter(OUTPUT_DIR / "Model_report.xlsx", engine="xlsxwriter") as writer:
    workbook = writer.book
    datasets = {
        "Main_measures": main_measures,
        "Effects": importance.loc[importance["importance"] > 1e-12],
        "Gini_over_time": gini_over_time_df,
        "Scorecard": scorecard,
        "Variable importance": raw_importance,
        "VIF_raw_features": raw_vif,
        "VIF_all_features": vif_all,
        "VIF_active_terms": vif_active,
        "Product_validation": product_metrics,
        "Submodel_comparison": submodel_metrics,
        "Calibration": test_calibration,
        "Variable": model_gini_vars,
    }
    header_format = workbook.add_format({"bold": True, "bg_color": "#D9EAF7"})
    percent_format = workbook.add_format({"num_format": "0.0%"})
    numeric_format = workbook.add_format({"num_format": "#,##0.000"})
    integer_format = workbook.add_format({"num_format": "#,##0"})
    percent_tokens = ("BR", "Share", "Gini", "PSI", "percent", "P. mode")
    integer_columns = {"All", "Bad", "Good", "GRP", "Number of distinct", "n_obs"}
    for sheet_name, dataframe in datasets.items():
        dataframe.to_excel(writer, sheet_name=sheet_name, index=False)
        worksheet = writer.sheets[sheet_name]
        for col_idx, column in enumerate(dataframe.columns):
            worksheet.write(0, col_idx, column, header_format)
            width = max(len(str(column)) + 2, 10)
            if not dataframe.empty:
                width = max(width, min(45, int(dataframe[column].astype(str).str.len().quantile(0.95)) + 2))
            cell_format = percent_format if any(token in str(column) for token in percent_tokens) else integer_format if column in integer_columns or str(column).endswith("_test") else numeric_format if pd.api.types.is_numeric_dtype(dataframe[column]) else None
            worksheet.set_column(col_idx, col_idx, width, cell_format)
        if dataframe.shape[0] > 0 and dataframe.shape[1] > 0:
            worksheet.autofilter(0, 0, dataframe.shape[0], dataframe.shape[1] - 1)
        worksheet.freeze_panes(1, 0)
    worksheet = writer.sheets["Gini_over_time"]
    if gini_over_time_df.shape[0] > 0:
        chart = workbook.add_chart({"type": "line"})
        chart.set_title({"name": "Gini over time"})
        chart.set_x_axis({"name": "Time", "num_font": {"rotation": 45}})
        chart.add_series({"name": "='Gini_over_time'!$B$1", "categories": f"='Gini_over_time'!$A$2:$A${gini_over_time_df.shape[0] + 1}", "values": f"='Gini_over_time'!$B$2:$B${gini_over_time_df.shape[0] + 1}"})
        chart.set_legend({"position": "bottom"})
        worksheet.insert_chart("D2", chart)

    used_sheet_names = set(writer.sheets.keys())
    title_format = workbook.add_format({"bold": True, "size": 18})
    for feature in final_variables[:MAX_VARIABLE_REPORT_SHEETS]:
        if feature not in variable_details:
            continue
        detail, time_summary = variable_details[feature]
        invalid = set("[]:*?/\\")
        clean = "".join("_" if char in invalid else char for char in str(feature))
        clean = clean[:31] or "Sheet"
        sheet_name = clean
        counter = 1
        while sheet_name in used_sheet_names:
            suffix = f"_{counter}"
            sheet_name = clean[: 31 - len(suffix)] + suffix
            counter += 1
        used_sheet_names.add(sheet_name)
        detail.to_excel(writer, sheet_name=sheet_name, startrow=2, index=False)
        time_summary.to_excel(writer, sheet_name=sheet_name, startrow=2, startcol=8, index=False)
        worksheet = writer.sheets[sheet_name]
        worksheet.write(0, 0, f"Variable: {feature}", title_format)
        for col_idx, column in enumerate(detail.columns):
            worksheet.write(2, col_idx, column, header_format)
            worksheet.set_column(col_idx, col_idx, max(len(str(column)) + 2, 10))
        for col_idx, column in enumerate(time_summary.columns):
            worksheet.write(2, 8 + col_idx, column, header_format)
            worksheet.set_column(8 + col_idx, 8 + col_idx, max(len(str(column)) + 2, 10))
        worksheet.freeze_panes(3, 0)
        detail_rows = detail.shape[0]
        time_rows = time_summary.shape[0]
        if detail_rows and time_rows:
            first_data_row = 4
            groups = detail_rows
            years_per_group = int(time_rows / groups) if groups else 0
            if years_per_group:
                for metric_name, column_letter, anchor_offset in [("BR", "N", 6), ("Share", "O", 22)]:
                    chart = workbook.add_chart({"type": "line"})
                    chart.set_title({"name": metric_name})
                    chart.set_x_axis({"name": "Time", "num_font": {"rotation": 45}})
                    chart.set_legend({"position": "bottom"})
                    for idx in range(groups):
                        start = first_data_row + idx * years_per_group
                        end = start + years_per_group - 1
                        chart.add_series({"name": f"='{sheet_name}'!$I${start}", "categories": f"='{sheet_name}'!$J${first_data_row}:$J${first_data_row + years_per_group - 1}", "values": f"='{sheet_name}'!${column_letter}${start}:${column_letter}${end}"})
                    worksheet.insert_chart(f"A{detail_rows + anchor_offset}", chart)



### Przygotowanie segmentów


In [ ]:
scored = sample[[TARGET, PERIOD_COLUMN]].copy()
scored[PD_COLUMN] = full_probability
scored[SCORE_COLUMN] = full_score
scored["Time"] = scored[PERIOD_COLUMN].astype(str).str[:4]
rank_segment = pd.qcut(scored[PD_COLUMN].rank(method="first"), q=3, labels=False, duplicates="drop").astype(int)
scored["Segment"] = (3 - 1) - rank_segment
balance_column = "cross_app_loan_amount" if "cross_app_loan_amount" in sample.columns else "app_loan_amount"
scored["outstanding"] = sample[balance_column].fillna(sample["app_loan_amount"])
scored["outstanding_bad"] = scored["outstanding"] * y

numbers = scored.groupby("Segment").agg(**{"Min score": (SCORE_COLUMN, "min"), "Max score": (SCORE_COLUMN, "max"), "All": (TARGET, "size"), "Bad": (TARGET, "sum")}).reset_index()
numbers["BR"] = numbers["Bad"] / numbers["All"]
numbers["Share"] = numbers["All"] / numbers["All"].sum()
numbers = numbers[["Segment", "Min score", "Max score", "BR", "Share", "All", "Bad"]]
years = sorted(scored["Time"].unique())
segments = sorted(scored["Segment"].unique())
yearly_totals = scored.groupby("Time")[TARGET].count().rename("Year_All")
numbers_time = scored.groupby(["Segment", "Time"])[TARGET].agg(Bad="sum", All="count").reindex(pd.MultiIndex.from_product([segments, years], names=["Segment", "Time"]), fill_value=0).reset_index().merge(yearly_totals, on="Time", how="left")
numbers_time["BR"] = np.where(numbers_time["All"] > 0, numbers_time["Bad"] / numbers_time["All"], np.nan)
numbers_time["Share"] = numbers_time["All"] / numbers_time["Year_All"]
numbers_time = numbers_time[["Segment", "Time", "BR", "Share", "All", "Bad"]]
balances = scored.groupby("Segment").agg(**{"Min score": (SCORE_COLUMN, "min"), "Max score": (SCORE_COLUMN, "max"), "outstanding": ("outstanding", "sum"), "outstanding_bad": ("outstanding_bad", "sum")}).reset_index()
balances["BRBal"] = balances["outstanding_bad"] / balances["outstanding"]
balances["Balance share"] = balances["outstanding"] / balances["outstanding"].sum()
balances = balances[["Segment", "Min score", "Max score", "BRBal", "Balance share", "outstanding", "outstanding_bad"]]
yearly_balance_totals = scored.groupby("Time")["outstanding"].sum().rename("Year_Balance")
balances_time = scored.groupby(["Segment", "Time"]).agg(outstanding=("outstanding", "sum"), outstanding_bad=("outstanding_bad", "sum")).reindex(pd.MultiIndex.from_product([segments, years], names=["Segment", "Time"]), fill_value=0).reset_index().merge(yearly_balance_totals, on="Time", how="left")
balances_time["BRBal"] = np.where(balances_time["outstanding"] > 0, balances_time["outstanding_bad"] / balances_time["outstanding"], np.nan)
balances_time["Balance share"] = balances_time["outstanding"] / balances_time["Year_Balance"]
balances_time = balances_time[["Segment", "Time", "BRBal", "Balance share", "outstanding", "outstanding_bad"]]



### Zapis Segments3_report.xlsx z wykresami


In [ ]:
with pd.ExcelWriter(OUTPUT_DIR / "Segments3_report.xlsx", engine="xlsxwriter") as writer:
    workbook = writer.book
    header_format = workbook.add_format({"bold": True, "bg_color": "#D9EAF7"})
    for sheet_name, left, right in [("Numbers", numbers, numbers_time), ("Balances", balances, balances_time)]:
        left.to_excel(writer, sheet_name=sheet_name, startrow=2, index=False)
        right.to_excel(writer, sheet_name=sheet_name, startrow=2, startcol=8, index=False)
        worksheet = writer.sheets[sheet_name]
        for col_idx, column in enumerate(left.columns):
            worksheet.write(2, col_idx, column, header_format)
            worksheet.set_column(col_idx, col_idx, max(len(str(column)) + 2, 10))
        for col_idx, column in enumerate(right.columns):
            worksheet.write(2, 8 + col_idx, column, header_format)
            worksheet.set_column(8 + col_idx, 8 + col_idx, max(len(str(column)) + 2, 10))
        worksheet.freeze_panes(3, 0)
        segments_count = left.shape[0]
        years_per_segment = int(right.shape[0] / segments_count) if segments_count else 0
        if years_per_segment:
            for title, column, anchor in [("Bad rate", "K", "A8"), ("Share", "L", "A24")]:
                chart = workbook.add_chart({"type": "line"})
                chart.set_title({"name": title})
                chart.set_x_axis({"name": "Time", "num_font": {"rotation": 45}})
                chart.set_legend({"position": "bottom"})
                for idx in range(segments_count):
                    start = 4 + idx * years_per_segment
                    end = start + years_per_segment - 1
                    chart.add_series({"name": f"='{sheet_name}'!$I${start}", "categories": f"='{sheet_name}'!$J$4:$J${3 + years_per_segment}", "values": f"='{sheet_name}'!${column}${start}:${column}${end}"})
                worksheet.insert_chart(anchor, chart)



### Zapis scoring_code.sas (scorecard PSC)


In [ ]:
with open(SAS_PATH, "w", encoding="utf-8") as file:
    file.write("proc sql; \n")
    file.write("create table  &zbior._score as \n")
    file.write("select indataset.*  \n")
    for var in vars_sc_num:
        scv = scorecard_scoring[scorecard_scoring["Variable"] == var].copy().reset_index()
        file.write(", case \n")
        for i in range(scv.shape[0]):
            war = scv["Condition"][i]
            score = scv["Score"][i]
            if war.count("<") == 2:
                file.write("when " + war.rsplit("<", 1)[0] + " and " + war.rsplit("<=", 1)[1] + " then " + str(score) + " \n")
            if war.count("<") == 1 and war.count("<>") == 0:
                file.write("when " + war + " then " + str(score) + " \n")
            if war.count("= " + SYMBOL_MISSING) == 1:
                file.write("when " + war.split(" ")[0] + " is null then " + str(score) + " \n")
            if war.count("<> " + SYMBOL_MISSING) == 1:
                file.write("when " + war.split(" ")[0] + " is not null then " + str(score) + " \n")
        score = scv["Score"][0]
        file.write("else " + str(score) + " end as PSC_" + var + " \n")
        file.write(" \n")
    for var in vars_sc_nom:
        scv = scorecard_scoring[scorecard_scoring["Variable"] == var].copy().reset_index()
        file.write(", case \n")
        index_other = 0
        for i in range(scv.shape[0]):
            war = scv["Condition"][i]
            score = scv["Score"][i]
            if war.count(",") == 0 and war.count(SYMBOL_OTHER) == 0:
                file.write("when " + var + " in (" + "'" + war + "'" + ") then " + str(score) + " \n")
            if war.count(",") > 0 and war.count(SYMBOL_OTHER) == 0:
                file.write("when " + var + " in (" + "'" + war.split(", ")[0] + "'")
                for j in range(war.count(",")):
                    file.write(", " + "'" + war.split(", ")[j + 1] + "'")
                file.write(") then " + str(score) + " \n")
            if war.count(SYMBOL_OTHER) == 1:
                index_other = i
        score = scv["Score"][index_other]
        file.write("else " + str(score) + " end as PSC_" + var + " \n")
        file.write(" \n")
    file.write("/* , 1/(1+exp(-(" + str(alpha) + "*(0.0")
    for var in vars_sc:
        file.write("+ calculated PSC_" + var)
    file.write("+(" + str(0) + ")))) as " + PD_COLUMN + " */ \n")
    file.write(" \n")
    file.write(", 0.0 \n")
    for var in vars_sc:
        file.write("+ calculated PSC_" + var + " ")
    file.write(" as SCORECARD_POINTS \n")
    file.write(" \n")
    file.write("from &zbior as indataset; \n")
    file.write("quit; \n")

scorecard_scoring.to_csv(OUTPUT_DIR / "sas_scorecard_points.csv", index=False)



### Podsumowanie uruchomienia


In [ ]:
test_metrics = metrics[-1]
print("PD Css Cross model: L1 regularized logistic regression")
print(f"Model ID: {MODEL_ID}")
print(f"Target: {TARGET}")
print(sample.attrs["sample_note"])
print(f"Train rows: {masks['train'].sum()}")
print(f"Validation rows: {masks['validation'].sum()}")
print(f"OOT test rows: {masks['test'].sum()}")
print(f"OOT test AUC: {test_metrics['auc']:.4f}")
print(f"OOT test Gini: {test_metrics['gini']:.4f}")
print(f"SAS scoring code: {SAS_PATH}")
print(f"Gini curves workbook: {GINI_CURVES_OUTPUT_PATH}")
print(f"Outputs written to: {OUTPUT_DIR}")

